# Treino de buracos na via (Roboflow) até o ONNX com passaporte

**Decisão que o modelo automatiza:** numa foto de rua, existe **buraco** no asfalto? Se sim, **ALERTA** para a
equipe de manutenção. Se a melhor candidata ficar na zona incerta, **VERIFICAR** (um humano olha). Caso
contrário, **OK**.

Fluxo deste notebook:

1. baixa o dataset do Roboflow Universe;
2. treina um YOLO11n e guarda o `best.pt`;
3. exporta o `best.pt` para `buraco_best.onnx` com o **passaporte** gravado dentro do arquivo;
4. escolhe uma foto de teste como exemplo e baixa `buraco_best.onnx` e `exemplo.jpg`.

Os dois arquivos vão para `web/public/exemplo/` e passam a ser o exemplo que o leitor carrega ao abrir.

Rode com **GPU T4** (Ambiente de execução → Alterar tipo) e cadastre `ROBOFLOW_API_KEY` em 🔑 *Secrets*.

## 0. Configuração

Tudo que depende do dataset está aqui. Para trocar de dataset, basta mudar este bloco: o resto do notebook
não cita nomes de classe. `TRADUCAO` é aplicada por nome (sem diferenciar maiúsculas); classe sem tradução
mantém o nome original. A regra de `DECISAO` é conferida contra as classes traduzidas antes de treinar.

In [ ]:
ROBOFLOW = {
    "workspace": "kartik-zvust",
    "projeto": "pothole-detection-yolo-v8",
    "versao": 1,  # None usa a versão mais recente
}
TRADUCAO = {"pothole": "buraco", "potholes": "buraco"}

NOME = "Buraco na via (YOLO11n)"
ARQUIVO = "buraco_best.onnx"
CONFIANCA_MINIMA = 0.35
DECISAO = {
    "alertar_se": ["buraco"],
    "limiar_alerta": 0.50,
    "zona_incerta": [0.30, 0.50],
    "mensagem": "Buraco na via",
}
TREINO = {"epochs": 50, "imgsz": 640, "patience": 15, "batch": 32}

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U ultralytics roboflow onnx onnxslim onnxruntime jsonschema

## 1. Código do passaporte

Clona só a pasta desta entrega do repositório. Se o repositório não estiver acessível, envie a pasta
`passaporte/` manualmente para `/content/entrega-03-onnx/passaporte/`.

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/m9tzin/lia1_2026_2.git"
PASTA = "Entregas - Matheus Marinho/entrega-03-onnx"
BASE = "/content/entrega-03-onnx"

if not os.path.isdir(f"{BASE}/passaporte"):
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", PASTA], check=True)
    os.symlink(f"/content/repo/{PASTA}", BASE)

sys.path.insert(0, BASE)
from passaporte.carimbar import ler
from passaporte.de_ultralytics import exportar, metricas_do_best
print("passaporte importado de", BASE)

## 2. Dataset do Roboflow

A chave vem dos Secrets do Colab (ou da variável de ambiente `ROBOFLOW_API_KEY` fora do Colab) e não
aparece no código.

In [ ]:
from roboflow import Roboflow

try:
    from google.colab import userdata
    CHAVE = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    CHAVE = os.environ.get("ROBOFLOW_API_KEY")
assert CHAVE, "Cadastre ROBOFLOW_API_KEY nos Secrets do Colab (ou como variável de ambiente)."
CHAVE = CHAVE.strip().strip('"').strip("'")  # espaço ou aspas coladas junto com a chave
assert not CHAVE.startswith("rf_"), "Essa é a Publishable Key. Use a Private API Key (Settings → API Keys)."

projeto = Roboflow(api_key=CHAVE).workspace(ROBOFLOW["workspace"]).project(ROBOFLOW["projeto"])
versoes = projeto.versions()
for v in versoes:
    print(f"versão {v.version}: {v.name}")

escolhida = projeto.version(ROBOFLOW["versao"]) if ROBOFLOW["versao"] else versoes[0]
print("\nusando:", escolhida.version, escolhida.name)

dataset = escolhida.download("yolov8", location="/content/dataset")
DATA_YAML = f"{dataset.location}/data.yaml"

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path(DATA_YAML).read_text())
nomes = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][i] for i in sorted(cfg["names"])]
classes_pt = [TRADUCAO.get(n.lower(), n) for n in nomes]
print("classes:", dict(zip(nomes, classes_pt)))

faltando = [c for c in DECISAO["alertar_se"] if c not in classes_pt]
assert not faltando, f"DECISAO cita {faltando}, que não existe em {classes_pt}. Ajuste TRADUCAO ou DECISAO."

for parte in ("train", "valid", "test"):
    pasta = Path(dataset.location) / parte / "images"
    if pasta.exists():
        print(f"{parte}: {len(list(pasta.glob('*')))} imagens")

## 3. Treino

`patience` interrompe se o *fitness* de validação parar de melhorar. Ao final, o Ultralytics grava em
`runs/detect/treino/weights/`:

- `last.pt`: pesos da última época;
- `best.pt`: pesos da época de **maior fitness** na validação. Na versão 8.4 o fitness de detecção é o
  mAP50-95. É esse arquivo que vai para produção.

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolo11n.pt")
modelo.train(data=DATA_YAML, project="/content/runs/detect", name="treino", exist_ok=True, **TREINO)

BEST = Path(modelo.trainer.best)
print("best.pt:", BEST)
print("métricas da época do best:", metricas_do_best(BEST))

In [ ]:
from IPython.display import Image, display
pasta = BEST.parent.parent
for arquivo in ("results.png", "confusion_matrix_normalized.png"):
    if (pasta / arquivo).exists():
        display(Image(filename=str(pasta / arquivo), width=900))

## 4. Exportar o best.pt com passaporte

A regra de decisão fica gravada no próprio modelo: quem abrir o `.onnx` em qualquer leitor que siga o
passaporte aplica a mesma política.

In [ ]:
DESTINO = f"/content/{ARQUIVO}"
caminho, passaporte = exportar(
    BEST,
    destino=DESTINO,
    classes=classes_pt,
    nome=NOME,
    dataset=f"roboflow {ROBOFLOW['workspace']}/{ROBOFLOW['projeto']} v{escolhida.version} ({escolhida.name})",
    confianca_minima=CONFIANCA_MINIMA,
    decisao=DECISAO,
)
print(caminho)

In [ ]:
# Prova de que tudo está dentro do arquivo: relê só o .onnx.
import json
print(json.dumps(ler(DESTINO), ensure_ascii=False, indent=2))

## 5. Conferência e foto de exemplo

Roda o ONNX pelo Ultralytics nas imagens de teste (o leitor web deve mostrar as mesmas caixas) e escolhe
como exemplo a foto com a detecção mais confiante de uma classe de alerta. Assim o site abre mostrando
um **ALERTA** real, e não uma foto qualquer.

In [ ]:
from PIL import Image as PILImage

testes = sorted((Path(dataset.location) / "test" / "images").glob("*")) or \
         sorted((Path(dataset.location) / "valid" / "images").glob("*"))
onnx_modelo = YOLO(DESTINO, task="detect")

# Uma foto por vez: o .onnx tem lote fixo 1 (como na Aula 13) e uma lista inteira viraria um lote só.
melhor, melhor_conf = None, 0.0
for caminho in testes:
    r = onnx_modelo.predict(str(caminho), conf=CONFIANCA_MINIMA, device="cpu", verbose=False)[0]
    alertas = [float(b.conf) for b in r.boxes if classes_pt[int(b.cls)] in DECISAO["alertar_se"]]
    if alertas and max(alertas) > melhor_conf:
        melhor, melhor_conf = str(caminho), max(alertas)

assert melhor, "nenhuma imagem de teste gerou alerta; confira o treino antes de publicar"
print(f"exemplo: {Path(melhor).name} ({melhor_conf:.0%})")

foto = PILImage.open(melhor).convert("RGB")
foto.thumbnail((1280, 1280))
foto.save("/content/exemplo.jpg", quality=88)
display(foto)

In [ ]:
from google.colab import files
files.download(DESTINO)
files.download("/content/exemplo.jpg")